In [9]:
import pandas as pd
import os
import json

In [10]:
# --------------------- JSON STATS SETTINGS --------------------- #
DATASET = "kdef"            # Options: "cxr" or "cub" or "kdef"
NET_CHOICE = "Transformer"  # Options: "Transformer", "Mamba", "CNN"
FROZEN = True               # Options: True or False, can be set to True only for NET_CHOICE=Transformer
BASE_WEIGHTS_DIR = "./drive_folder/Bridging_Human_and_Model_Attention_Explainability_Analysis_of_CNN_Mamba_and_ViT_Architectures_with_Gaze-Based_Validation"
data_folder_name = "CUB_200_2011" if DATASET == "cub" else DATASET.upper()
heatmap_file_name = f"heatmap_scores_frozen.json" if NET_CHOICE == "Transformer" and FROZEN else 'heatmap_scores_frozen.json'
file_path = os.path.join(BASE_WEIGHTS_DIR, NET_CHOICE, 'output_heatmaps', data_folder_name, heatmap_file_name)
print("Loading JSON from:", file_path)

Loading JSON from: ./drive_folder/Bridging_Human_and_Model_Attention_Explainability_Analysis_of_CNN_Mamba_and_ViT_Architectures_with_Gaze-Based_Validation/Transformer/output_heatmaps/KDEF/heatmap_scores_frozen.json


In [11]:
# Load your data 
with open(file_path, 'r') as f:
    data = json.load(f)

rows = []

for filename, content in data.items():
    # 1. Extract the shared metadata
    index_val = content.get('index')
    train_val = content.get('train')
    
    # 2. Iterate through the keys to find the explainability methods
    # We ignore 'index' and 'train' keys
    for key, value in content.items():
        print(key)

        if key not in ['index', 'train']:
            model = key
            print(model)

            if model == "ResNet50":
                for cam_method in value.keys():
                    metrics = value.get(cam_method)
                    row = {
                        'name': filename,
                        'index': index_val,
                        'train': train_val,
                        'model': model,
                        'explainability_method': cam_method,
                        'JSS': metrics.get('JSS'),
                        'Chi2': metrics.get('Chi2'),
                        'PCC': metrics.get('PCC')
                    }
                    rows.append(row)
            else:
                print(value)
                metrics = value
                
                row = {
                    'name': filename,
                    'index': index_val,
                    'train': train_val,
                    'model': model,
                    'explainability_method': "x",
                    'JSS': metrics.get('JSS'),
                    'Chi2': metrics.get('Chi2'),
                    'PCC': metrics.get('PCC')
                }
                rows.append(row)
            

# 4. Create the DataFrame
df = pd.DataFrame(rows)

# Optional: Set the column order exactly as requested
df = df[['name', 'index', 'train', 'explainability_method', 'JSS', 'Chi2', 'PCC']]

print(df.head())

index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.44376126600518706, 'Chi2': -0.022005861338957988, 'PCC': 0.18310864742025404}
index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.4828841324062355, 'Chi2': 0.1389415549396419, 'PCC': 0.5716106406454173}
index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.4124754082481267, 'Chi2': -0.10625929531507472, 'PCC': 0.19265888266065498}
index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.4243842361791961, 'Chi2': -0.07281790437260138, 'PCC': 0.1941796684239307}
index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.45105049545480846, 'Chi2': 0.027696408015218443, 'PCC': 0.42288867851816125}
index
train
Transformervit_base_patch16_224
Transformervit_base_patch16_224
{'JSS': 0.5203677136204544, 'Chi2': 0.23032271014367844, 'PCC': 0.36238025045055455}
index
train
Transformervit_bas

In [12]:
#mean scores divided between explainability method and train/test set
cols_to_mean = ['JSS', 'Chi2', 'PCC']

mean_scores = df.groupby(['explainability_method', 'train'])[cols_to_mean].mean()
std_scores = df.groupby(['explainability_method', 'train'])[cols_to_mean].std()

print(mean_scores)
print(std_scores)

                                  JSS      Chi2       PCC
explainability_method train                              
x                     False  0.438862 -0.019371  0.278823
                                  JSS      Chi2       PCC
explainability_method train                              
x                     False  0.047553  0.156701  0.152554


In [13]:
import pandas as pd

In [14]:
# --------------------- COMPARISON CSV STATS SETTINGS --------------------- #
import os

DATASET = "kdef"        # "cub" | "cxr" | "kdef"
OUTPUT_TYPE = "gaze"    # "gaze"     -> matches TABLE VII (model attention maps alone)
                        # "heatmaps" -> the image+overlay version

# One per-dataset block of TABLE VII = these 3 model pairs.
# Folder name -> paper name:  CNN = ResNet50, Transformer = ViT-base, Mamba = ViM-base
PAIRS = [("CNN", "Transformer"),
         ("CNN", "Mamba"),
         ("Transformer", "Mamba")]
NICE = {"CNN": "ResNet50", "Transformer": "ViT-base", "Mamba": "ViM-base"}


In [15]:
# Build one row per model pair (mean ± std over the test images), like TABLE VII.
table_rows = []
for FOLDER_1, FOLDER_2 in PAIRS:
    csv_file_path = f"heatmap_comparison_results/{FOLDER_1}_{FOLDER_2}_{DATASET}_{OUTPUT_TYPE}.csv"
    if not os.path.exists(csv_file_path):
        print(f"[skip] missing CSV: {csv_file_path}")
        continue
    df = pd.read_csv(csv_file_path).set_index('filename')
    mean = df.mean(numeric_only=True)
    std = df.std(numeric_only=True)
    table_rows.append({
        "Model":          NICE.get(FOLDER_1, FOLDER_1),
        "Model_2":        NICE.get(FOLDER_2, FOLDER_2),
        "Dataset":        DATASET.upper(),
        "Jensen-Shannon": f"{mean['JSS']:.4f} ± {std['JSS']:.4f}",
        "Chi-square":     f"{mean['Chi2']:.4f} ± {std['Chi2']:.4f}",
        "PCC":            f"{mean['PCC']:.4f} ± {std['PCC']:.4f}",
        "n":              len(df),
    })


In [16]:
table = pd.DataFrame(table_rows)
pd.set_option("display.max_colwidth", None)
print(f"TABLE VII rows  (DATASET={DATASET.upper()}, OUTPUT_TYPE={OUTPUT_TYPE})\n")
print(table.to_string(index=False))


TABLE VII rows  (DATASET=KDEF, OUTPUT_TYPE=gaze)

   Model  Model_2 Dataset  Jensen-Shannon       Chi-square              PCC   n
ResNet50 ViT-base    KDEF 0.6731 ± 0.0449  0.6211 ± 0.1052  0.7015 ± 0.0941 120
ResNet50 ViM-base    KDEF 0.3957 ± 0.0164 -0.1635 ± 0.0587 -0.0420 ± 0.2002 120
ViT-base ViM-base    KDEF 0.4416 ± 0.0501 -0.0217 ± 0.1563 -0.0308 ± 0.1858 120
